In [6]:
import pymysql
print("ok")

ok


In [7]:
from sqlalchemy.engine import URL
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

# Use password and user from SQL

## Establish a connection between Python and the Sakila database.

In [8]:
load_dotenv() #This part is to avoid sharing my password in the repo

url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"), #Because of the way the password is
    host=os.getenv("DB_HOST"),
    database=os.getenv("DB_NAME")
)

engine = create_engine(url)


In [9]:
df_test = pd.read_sql("SELECT * FROM rental LIMIT 5;", engine)
df_test

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


## Write a Python function called rentals_month 

Write a Python function called rentals_month that retrieves rental data for a given month and year (passed as parameters) from the Sakila database as a Pandas DataFrame. The function should take in three parameters:

- engine: an object representing the database connection engine to be used to establish a connection to the Sakila database.

- month: an integer representing the month for which rental data is to be retrieved.

- year: an integer representing the year for which rental data is to be retrieved.

The function should execute a SQL query to retrieve the rental data for the specified month and year from the rental table in the Sakila database, and return it as a pandas DataFrame.

In [10]:
def rentals_month(engine, month, year):

    query = f"""
    SELECT *
    FROM rental
    WHERE MONTH(rental_date) = {month}
    AND YEAR(rental_date) = {year}
    """

    return pd.read_sql(query, engine)

### Test example with may 2005

In [11]:
may = rentals_month(engine, 5, 2005)
may.head()

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


## Develop a Python function called rental_count_month 

Develop a Python function called rental_count_month that takes the DataFrame provided by rentals_month as input along with the month and year and returns a new DataFrame containing the number of rentals made by each customer_id during the selected month and year.

The function should also include the month and year as parameters and use them to name the new column according to the month and year, for example, if the input month is 05 and the year is 2005, the column name should be "rentals_05_2005".

Hint: Consider making use of pandas groupby()

In [15]:
def rental_count_month(df, month, year):

    column_name = f"rentals_{month:02d}_{year}" #02d means to have 2 digits

    result = df.groupby("customer_id").size().reset_index(name = column_name)
    
    return result

### Test example with may 2005

In [16]:
may_counts = rental_count_month(may, 5, 2005)
may_counts.head()

,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


## Create a Python function called compare_rentals 

Create a Python function called compare_rentals that takes two DataFrames as input containing the number of rentals made by each customer in different months and years. 

The function should return a combined DataFrame with a new 'difference' column, which is the difference between the number of rentals in the two months.

In [17]:
def compare_rentals(df1, df2):
    comparison = pd.merge(df1, df2, on="customer_id",how="outer").fillna(0)

    col1 = df1.columns[1]
    col2 = df2.columns[1]

    comparison["diference"]= comparison[col2] - comparison[col1]
    

    return comparison

### Test example with may 2005 vs june 2005

In [18]:
may = rentals_month(engine, 5, 2005)
june = rentals_month(engine, 6, 2005)

may_counts = rental_count_month(may, 5, 2005)
june_counts = rental_count_month(june, 6, 2005)

comparison = compare_rentals(may_counts, june_counts)

comparison.head()

,customer_id,rentals_05_2005,rentals_06_2005,diference
0,1,2.0,7.0,5.0
1,2,1.0,1.0,0.0
2,3,2.0,4.0,2.0
3,4,0.0,6.0,6.0
4,5,3.0,5.0,2.0
